This code converts normalized gesture sequences into fixed-length FFT-based frequency features and saves them with both 18-class and binary labels for model training.

In [1]:
import pandas as pd
import numpy as np
from scipy.fft import fft
import os
from pathlib import Path
import matplotlib.pyplot as plt


NOTEBOOK_DIR = Path.cwd()

def find_repo_root(start_path: Path):
    cur = start_path
    for _ in range(10):
        if (cur / ".git").exists() or (cur / "data").exists():
            return cur
        cur = cur.parent
    return start_path

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)


DATA_DIR = REPO_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"

train_path = PROC_DIR / "train_normalized.parquet"
test_path  = PROC_DIR / "test_normalized.parquet"


output_dir = PROC_DIR / "Processed_FFT"
output_dir.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Train in :", train_path)
print("Test in  :", test_path)
print("Output   :", output_dir)


train_DataFrame = pd.read_parquet(train_path)
test_DataFrame  = pd.read_parquet(test_path)


IMU_COLS = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
THM_COLS = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
TOF_COLS = [c for c in train_DataFrame.columns if 'tof' in c]

Total_Cols = sorted(IMU_COLS + THM_COLS + TOF_COLS)

NO_TOF_COLS = sorted(IMU_COLS + THM_COLS)

print(f"Total processing cols count: {len(Total_Cols)}")

# 
# FFT function
# 
def process_one_sequence_FFT(data_matrix):
    target_len = 64
    current_len = len(data_matrix)

    if current_len >= target_len:
        data_matrix = data_matrix[:target_len, :]
    else:
        pad_len = target_len - current_len
        data_matrix = np.pad(data_matrix, ((0, pad_len), (0, 0)), mode='constant')

    fft_result = fft(data_matrix, axis=0)
    fft_magnitude = np.abs(fft_result)
    fft_final = fft_magnitude[:32, :]
    return fft_final.flatten()

# 
# get label
#
gesture_classes = sorted(train_DataFrame['gesture'].dropna().unique().tolist())
print("Number of gesture classes in train:", len(gesture_classes))
print("Classes:", gesture_classes)

gesture_to_idx = {g: i for i, g in enumerate(gesture_classes)}

BFRB_Gestures = [
    'Cheek - pinch skin',
    'Forehead - pull hairline',
    'Neck - scratch',
    'Neck - pinch skin',
    'Eyelash - pull hair',
    'Eyebrow - pull hair',
    'Forehead - scratch',
    'Above ear - pull hair'
]

# 
# Get sequence_id
# 
train_Id = train_DataFrame['sequence_id'].unique()
test_Id  = test_DataFrame['sequence_id'].unique()

data_train_list = []
label_train_18_list = []
label_train_binary_list = []

data_test_list = []
label_test_18_list = []
label_test_binary_list = []

# 
# FFT for train
#
for seq_id in train_Id:
    current_seq_data = train_DataFrame[train_DataFrame['sequence_id'] == seq_id]
    raw_values = current_seq_data[Total_Cols].values
    processed_data = process_one_sequence_FFT(raw_values)
    data_train_list.append(processed_data)

    gesture_name = current_seq_data['gesture'].iloc[0]
    label18 = gesture_to_idx.get(gesture_name, -1)
    label_train_18_list.append(label18)

    label_binary = 1 if gesture_name in BFRB_Gestures else 0
    label_train_binary_list.append(label_binary)

data_train = np.array(data_train_list)
label_train_18 = np.array(label_train_18_list)
label_train_binary = np.array(label_train_binary_list)

#
# FFT for test
#
for seq_id in test_Id:
    current_seq_data = test_DataFrame[test_DataFrame['sequence_id'] == seq_id]
    raw_values = current_seq_data[Total_Cols].values
    processed_data = process_one_sequence_FFT(raw_values)
    data_test_list.append(processed_data)

    gesture_name = current_seq_data['gesture'].iloc[0]
    label18 = gesture_to_idx.get(gesture_name, -1)
    label_test_18_list.append(label18)

    label_binary = 1 if gesture_name in BFRB_Gestures else 0
    label_test_binary_list.append(label_binary)

data_test = np.array(data_test_list)
label_test_18 = np.array(label_test_18_list)
label_test_binary = np.array(label_test_binary_list)

# 
# Save the FFT result
#
np.save(output_dir / "data_train.npy", data_train)
np.save(output_dir / "label_train_18.npy", label_train_18)
np.save(output_dir / "label_train_binary.npy", label_train_binary)

np.save(output_dir / "data_test.npy", data_test)
np.save(output_dir / "label_test_18.npy", label_test_18)
np.save(output_dir / "label_test_binary.npy", label_test_binary)

np.save(output_dir / "gesture_classes.npy", np.array(gesture_classes, dtype=object))

print("OVER!")


Repo root: E:\A1_last_version
Train in : E:\A1_last_version\data\processed\train_normalized.parquet
Test in  : E:\A1_last_version\data\processed\test_normalized.parquet
Output   : E:\A1_last_version\data\processed\Processed_FFT
Total processing cols count: 332
Number of gesture classes in train: 18
Classes: ['Above ear - pull hair', 'Cheek - pinch skin', 'Drink from bottle/cup', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Feel around in tray and pull out an object', 'Forehead - pull hairline', 'Forehead - scratch', 'Glasses on/off', 'Neck - pinch skin', 'Neck - scratch', 'Pinch knee/leg skin', 'Pull air toward your face', 'Scratch knee/leg skin', 'Text on phone', 'Wave hello', 'Write name in air', 'Write name on leg']
OVER!


This code selects one BFRB and one non-BFRB sequence, extracts FFT features separately for IMU, THM, and TOF modalities, and generates comparison plots saved to the plots directory.

In [2]:

import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()

def find_repo_root(start_path: Path):
    cur = start_path
    for _ in range(10):
        if (cur / ".git").exists() or (cur / "data").exists():
            return cur
        cur = cur.parent
    return start_path

REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
PROC_DIR = REPO_ROOT / "data" / "processed"

train_path = PROC_DIR / "train_normalized.parquet"
out_base  = REPO_ROOT / "plots"
fig_dir   = out_base / "bfrb_vs_nonbfrb_modality_plots"
fig_dir.mkdir(parents=True, exist_ok=True)

print("Train parquet:", train_path)
print("Figure dir   :", fig_dir)

train_df = pd.read_parquet(train_path)

IMU_COLS = ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
THM_COLS = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
TOF_COLS = [c for c in train_df.columns if str(c).lower().startswith("tof_")]  

print("IMU cols:", len(IMU_COLS))
print("THM cols:", len(THM_COLS))
print("TOF cols:", len(TOF_COLS))

# 
# BFRB Labels
#
BFRB_Gestures = [
    'Cheek - pinch skin',
    'Forehead - pull hairline',
    'Neck - scratch',
    'Neck - pinch skin',
    'Eyelash - pull hair',
    'Eyebrow - pull hair',
    'Forehead - scratch',
    'Above ear - pull hair'
]

train_df["_is_bfrb"] = train_df["gesture"].apply(lambda g: 1 if g in BFRB_Gestures else 0)

def process_one_sequence_FFT_by_cols(seq_df, cols, target_len=64):
    data_matrix = seq_df[cols].values
    cur_len = len(data_matrix)

    if cur_len >= target_len:
        data_matrix = data_matrix[:target_len, :]
    else:
        pad_len = target_len - cur_len
        data_matrix = np.pad(data_matrix, ((0, pad_len), (0, 0)), mode='constant')

    fft_result = fft(data_matrix, axis=0)
    fft_mag = np.abs(fft_result)
    fft_half = fft_mag[:32, :]          
    return fft_half.flatten()

# 
# Choose a Non_BFBR and a BFBR's sequence_id
# 
seq_label = train_df.groupby("sequence_id")["_is_bfrb"].first()

non_ids = seq_label[seq_label == 0].index.values
bfrb_ids = seq_label[seq_label == 1].index.values

if len(non_ids) == 0 or len(bfrb_ids) == 0:
    raise ValueError("Train split do not incluse BFRB and non-BFRB sequence， can not draw the picture.")

seq_non = non_ids[0]
seq_bfrb = bfrb_ids[0]

print("Chosen non-BFRB seq_id:", seq_non)
print("Chosen BFRB    seq_id:", seq_bfrb)

seq_non_df  = train_df[train_df["sequence_id"] == seq_non]
seq_bfrb_df = train_df[train_df["sequence_id"] == seq_bfrb]

# 
# calculate three different FFT fitures
#
feat_non_imu = process_one_sequence_FFT_by_cols(seq_non_df, IMU_COLS)
feat_non_thm = process_one_sequence_FFT_by_cols(seq_non_df, THM_COLS)
feat_non_tof = process_one_sequence_FFT_by_cols(seq_non_df, TOF_COLS)

feat_bfrb_imu = process_one_sequence_FFT_by_cols(seq_bfrb_df, IMU_COLS)
feat_bfrb_thm = process_one_sequence_FFT_by_cols(seq_bfrb_df, THM_COLS)
feat_bfrb_tof = process_one_sequence_FFT_by_cols(seq_bfrb_df, TOF_COLS)


k = 200  

def plot_three_modalities(one_title, f_imu, f_thm, f_tof, save_path):
    imu_s = f_imu[:min(k, len(f_imu))]
    thm_s = f_thm[:min(k, len(f_thm))]
    tof_s = f_tof[:min(k, len(f_tof))]

    y_min = min(imu_s.min(), thm_s.min(), tof_s.min())
    y_max = max(imu_s.max(), thm_s.max(), tof_s.max())

    plt.figure(figsize=(10, 4))
    plt.plot(range(len(imu_s)), imu_s, label="IMU-only FFT (first 200 dims)")
    plt.plot(range(len(thm_s)), thm_s, label="THM-only FFT (first 200 dims)")
    plt.plot(range(len(tof_s)), tof_s, label="TOF-only FFT (first 200 dims)")
    plt.title(one_title)
    plt.xlabel("Feature Index (first 200)")
    plt.ylabel("FFT Magnitude")
    plt.ylim(y_min, y_max)
    plt.legend()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

# Non-BFRB picture
p1 = fig_dir / f"nonbfrb_seq_{seq_non}_imu_thm_tof.png"
plot_three_modalities(
    one_title=f"Non-BFRB Sequence {seq_non} | IMU vs THM vs TOF (Flattened FFT, first 200 dims)",
    f_imu=feat_non_imu, f_thm=feat_non_thm, f_tof=feat_non_tof,
    save_path=p1
)

# BFRB picture
p2 = fig_dir / f"bfrb_seq_{seq_bfrb}_imu_thm_tof.png"
plot_three_modalities(
    one_title=f"BFRB Sequence {seq_bfrb} | IMU vs THM vs TOF (Flattened FFT, first 200 dims)",
    f_imu=feat_bfrb_imu, f_thm=feat_bfrb_thm, f_tof=feat_bfrb_tof,
    save_path=p2
)

print("Saved 2 modality-comparison plots:")
print(p1)
print(p2)


Train parquet: E:\A1_last_version\data\processed\train_normalized.parquet
Figure dir   : E:\A1_last_version\plots\bfrb_vs_nonbfrb_modality_plots
IMU cols: 7
THM cols: 5
TOF cols: 320
Chosen non-BFRB seq_id: SEQ_000016
Chosen BFRB    seq_id: SEQ_000007
Saved 2 modality-comparison plots:
E:\A1_last_version\plots\bfrb_vs_nonbfrb_modality_plots\nonbfrb_seq_SEQ_000016_imu_thm_tof.png
E:\A1_last_version\plots\bfrb_vs_nonbfrb_modality_plots\bfrb_seq_SEQ_000007_imu_thm_tof.png
